# MRT2 WebUI — Google Colab Demo

Run the **[mrt2-webui](https://github.com/yhs3048/mrt2-webui)** realtime music generation UI on a free Colab GPU.

Stream AI-generated music in your browser and steer the style with **text**, **images**, **video**, or your **screen** — all powered by [Magenta RT2](https://github.com/magenta/magenta-realtime).

---

## Before you start

1. Set the runtime to GPU: **Runtime → Change runtime type → Hardware accelerator → GPU** (T4 is fine).
2. Run the cells top to bottom.
3. The last cell prints a public **`https://...trycloudflare.com`** URL — open it in a new tab.

> **Note:** A free T4 can run `mrt2_small` (230M). The larger `mrt2_base` (2.4B) may generate slower than realtime on a T4, causing playback gaps. Start with `mrt2_small`.

> **Realtime may not be smooth on Colab.** Audio is generated on the Colab GPU and streamed to your browser through a public tunnel (cloudflared). The extra network hops add latency and jitter on top of generation time, so playback can stutter even when the model itself keeps up — especially on a shared free T4. For the smoothest, true-realtime experience, run the WebUI locally on your own GPU. On Colab, prefer `mrt2_small` and a smaller **Frames / chunk** to reduce gaps.

## 1. Check the GPU

If this errors or shows no GPU, switch the runtime to GPU first (see above).

In [ ]:
!nvidia-smi

## 2. Clone the repo and install dependencies

Installs Magenta RT2 (JAX + CUDA 12 backend) and the WebUI requirements. This takes a few minutes the first time.

In [ ]:
import os

# Clone the WebUI (skip if it already exists)
if not os.path.isdir("/content/mrt2-webui"):
    !git clone https://github.com/yhs3048/mrt2-webui.git /content/mrt2-webui

# Magenta RT2 core (JAX backend) + CUDA 12 JAX wheel for the Colab GPU
!pip install -q "magenta-rt" "jax[cuda12]"

# WebUI server dependencies (fastapi, uvicorn, websockets, transformers, ...)
!pip install -q -r /content/mrt2-webui/requirements.txt

print("\n\u2713 Install complete.")

## 3. Download the model weights

Sets `MAGENTA_HOME` and downloads the shared resources (MusicCoCa + SpectroStream) and the raw `mrt2_small` checkpoint that the JAX engine loads.

To also use `mrt2_base`, uncomment the last line (large download, slow on a T4).

In [ ]:
import os

# Where Magenta RT2 stores all weights/resources
os.environ["MAGENTA_HOME"] = "/content/magenta_home"

# Shared resources: MusicCoCa (style) + SpectroStream (codec)
!mrt models init

# Raw safetensors checkpoint used by the JAX engine (MagentaRT2Jax)
!mrt checkpoints download mrt2_small.safetensors

# Optional: the larger 2.4B model (big download; may be slower than realtime on T4)
# !mrt checkpoints download mrt2_base.safetensors

print("\n\u2713 Models ready under", os.environ["MAGENTA_HOME"])

## 4. Get a public URL (cloudflared)

Colab can't expose a raw port to your browser, and the WebUI needs **WebSocket** + **HTTPS** (for screen / camera capture). We use `cloudflared` to open a free public HTTPS tunnel to the server.

This cell just downloads the `cloudflared` binary.

In [ ]:
import os

if not os.path.exists("/content/cloudflared"):
    !wget -q -O /content/cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
    !chmod +x /content/cloudflared

!/content/cloudflared --version

## 5. Launch the WebUI

Starts `webui.py` on port 8000 in the background, then opens the cloudflared tunnel and prints the public URL.

Open the **`https://...trycloudflare.com`** link in a new tab, pick **mrt2_small**, click **Load model**, then **Play**.

> First model load takes ~30–60s (JAX compiles the graph). Watch the `webui.log` output below for `model ready`.

In [ ]:
import os, re, time, subprocess, threading

os.environ["MAGENTA_HOME"] = "/content/magenta_home"
PORT = 8000

# 1) Start the WebUI server (background), logging to webui.log
server = subprocess.Popen(
    ["python", "webui.py", "--host", "127.0.0.1", "--port", str(PORT)],
    cwd="/content/mrt2-webui",
    stdout=open("/content/webui.log", "w"),
    stderr=subprocess.STDOUT,
    env={**os.environ},
)
print(f"WebUI starting (pid={server.pid}) on port {PORT} ...")

# 2) Start cloudflared and capture the public URL it prints
tunnel = subprocess.Popen(
    ["/content/cloudflared", "tunnel", "--no-autoupdate", "--url", f"http://127.0.0.1:{PORT}"],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

public_url = None
deadline = time.time() + 60
url_re = re.compile(r"https://[-a-z0-9]+\.trycloudflare\.com")
while time.time() < deadline:
    line = tunnel.stdout.readline()
    if not line:
        if tunnel.poll() is not None:
            break
        continue
    print(line, end="")
    m = url_re.search(line)
    if m:
        public_url = m.group(0)
        break

# Keep draining cloudflared output so its buffer doesn't fill up
def _drain(p):
    for _ in iter(p.stdout.readline, ""):
        pass

threading.Thread(target=_drain, args=(tunnel,), daemon=True).start()

print("\n" + "=" * 60)
if public_url:
    print("  Open this URL in a new tab:")
    print("  " + public_url)
else:
    print("  Could not detect the cloudflared URL. Check the output above.")
print("=" * 60)

### Server logs (optional)

Run this to watch the server log — useful to confirm the model loaded or to debug errors.

In [ ]:
!tail -n 40 /content/webui.log

---

## Tips & troubleshooting

- **Realtime is hard on Colab:** Even when the GPU generates fast enough, audio is streamed to your browser through a public tunnel (cloudflared), which adds network latency and jitter. This alone can cause stutter/gaps. The Colab setup is best treated as a **demo** — for true realtime, run the WebUI locally on your own GPU.
- **No sound / gaps:** A T4 may not keep up in realtime. Use `mrt2_small`, and in the UI lower **Frames / chunk** under *Settings*.
- **Screen / camera capture not working:** Make sure you opened the `https://` cloudflared URL (not an `http://` link). Browsers only allow capture over HTTPS.
- **URL not printed:** Re-run the launch cell; the cloudflared URL can take up to a minute to appear.
- **Session ended:** Colab disconnects idle runtimes. Re-run the launch cell to get a fresh URL.
- **Stop everything:** Runtime → Disconnect and delete runtime, or restart the runtime.

Built on [Magenta RT2](https://github.com/magenta/magenta-realtime) by Google Magenta · WebUI by [yhs3048](https://github.com/yhs3048) (MARG, SNU).